In [1]:
import os, sys
from pathlib import Path
import pandas as pd
import subprocess

In [2]:
sys.path.append("../../../training_data")

In [3]:
from utils.utils import Cif

In [4]:
with open("../../../training_data/8.Apos/Extra_set/features.pkl", "rb") as f:
    extras_featuresd = pd.read_pickle(f)

len(extras_featuresd), extras_featuresd

(24,
 {'8sgj':     Residues                                                          \
           pdb label_entity_id label_asym_id label_seq_id auth_asym_id   
  0       8sgj               1             A           52            A   
  1       8sgj               1             A           53            A   
  2       8sgj               1             A           54            A   
  3       8sgj               1             A           55            A   
  4       8sgj               1             A           56            A   
  ..       ...             ...           ...          ...          ...   
  746     8sgj               1             A          941            A   
  747     8sgj               1             A          942            A   
  748     8sgj               1             A          943            A   
  749     8sgj               1             A          944            A   
  750     8sgj               1             A          945            A   
  
                      

In [5]:
remaining = []

for pdb, feats in extras_featuresd.items():
    outdir = Path(pdb)
    if not outdir.exists():
        remaining.append(pdb)

len(remaining), remaining

(18,
 ['8szq',
  '5e1i',
  '6gx3',
  '2gk9',
  '2gs3',
  '7v37',
  '9di9',
  '5c97',
  '2ojx',
  '6uen',
  '3pfv',
  '8y6y',
  '4n22',
  '9di2',
  '9fsk',
  '5e97',
  '9ouk',
  '8yf0'])

# Sequences/PSSMs

In [6]:
for pdb in sorted(remaining):
    path = Path("ncbi_psiblast_pssms") / pdb
    path.mkdir(exist_ok = True, parents=True)

    feats = extras_featuresd[pdb]
    chain = feats[('Residues', 'auth_asym_id')].unique().item()
    name = f"{pdb}_{chain}"


    fastaf = path / f"{pdb}.fasta"
    if not fastaf.exists():
        seq = (
            pd.DataFrame(
                Cif(
                    pdb, 
                    filename = Path("..") / "structures" / f"{pdb.lower()}.cif"
                ).cif.data["_entity_poly"], dtype=str
            )
            .query(f"entity_id == '{feats[('Residues', 'label_entity_id')].unique().item()}'")
            ["pdbx_seq_one_letter_code_can"].item()
            .replace("\n", "")
        )
        
        with open(fastaf, "w") as f:
            f.write(f">{name}\n")
            f.write(seq)
        print(fastaf)

    pssmf = path / f"{name}.pssm"
    if not (pssmf).exists():
        for f in path.glob("*.asn"):
            subprocess.run(f"""
psiblast \
-in_pssm {f} \
-subject {fastaf} \
-num_iterations 1 \
-out_ascii_pssm {pssmf} \
-out {fastaf.with_suffix(".out")}""",
                shell=True, check=True
            )
            print(pssmf)

# Make predictions

Edits throughout to fix:
- Hardcoded paths
- Pass locations of ProtT5 and the nr database
- Use existing .pssm

In [7]:
for pdb, feats in extras_featuresd.items():
    # if pdb == "7sns": continue        
    path = Path("../../other_tools/AlloFusion/AlloFusion/Case Study").resolve()
    path.mkdir(exist_ok = True)
    try:
        outdir = Path(pdb)
        if not outdir.exists():
            chain = feats[('Residues', 'auth_asym_id')].unique().item()
            
            origpdbf = Path(f"../structures/{pdb}.pdb").resolve()
    
            seq = (
                pd.DataFrame(
                    Cif(pdb, origpdbf.with_suffix(".cif")).cif.data["_entity_poly"], dtype=str
                )
                .query(f"entity_id == '{feats[('Residues', 'label_entity_id')].unique().item()}'")
                ["pdbx_seq_one_letter_code_can"].item()
                .replace("\n", "")
            )

            pssmf = Path("ncbi_psiblast_pssms") / pdb / f"{pdb}_{chain}.pssm"
            if pssmf.exists():
                print(pdb)
                (path / f"{pdb}_{chain}.pssm").symlink_to(pssmf.resolve())
            else:
                print(pdb, "missing pssm")
                continue
            
            pdbf = path / f"{pdb}.pdb"
            if not pdbf.exists():
                pdbf.symlink_to(origpdbf)
        
            subprocess.run(f"python AlloFusionMain.py --PDBID {pdb} --CHAIN {chain} --SEQ {seq} --huggingface_dir /data/fnerin/huggingface --nr_database /data/fnerin/nr_database/nr", cwd=path.parent, shell=True, check=True)

            path.rename(outdir)
            
    except Exception as e:
        print(f"ERROR: ", pdb)
        print(e)
        path.rename(f"{pdb}_error")
        continue

8szq


2026-05-06 12:52:33.104916: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-06 12:52:33.146744: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Embedding done!
PSSM done!
Bio done!
13/32 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step

29/32 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step

...

2026-05-06 18:48:18.019373: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-06 18:48:18.063313: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Embedding done!
PSSM done!
Bio done!
11/24 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step

21/24 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step

24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step


# Process

In [8]:
results = {}

for pdb, feats in extras_featuresd.items():
    # if pdb == "7sns": continue
    chain = feats[('Residues', 'auth_asym_id')].unique().item()
    resf = f"{pdb}/{pdb}_allosteric_residues.txt"
    if os.path.isfile(resf):
        with open(resf) as f:
            txt = f.read()
        chain = txt.split("Chain", 1)[1].strip().split()[0]
        resids = [x for x in txt.split("resid", 1)[1].replace("(", "").replace(")", "").replace(",", " ").split() if x.isdigit()]   

        results[pdb.lower()] = {"pocket": {"residues": (
            pd.DataFrame({"auth_asym_id": [chain]*len(resids), "auth_seq_id": resids}, dtype=str)
            .merge(Cif(pdb, f"../structures/{pdb}.cif").residues)
            [["auth_asym_id", "auth_seq_id"]]
        )}}

len(results), results

(24,
 {'8sgj': {'pocket': {'residues':   auth_asym_id auth_seq_id
    0            A         451
    1            A         810}},
  '7l6r': {'pocket': {'residues':   auth_asym_id auth_seq_id
    0            A        6878
    1            A        6880
    2            A        6968
    3            A        7000}},
  '6yhr': {'pocket': {'residues':    auth_asym_id auth_seq_id
    0             A         577
    1             A         581
    2             A         704
    3             A         713
    4             A         720
    5             A         721
    6             A         730
    7             A         731
    8             A         854
    9             A         888
    10            A         981}},
  '7xlq': {'pocket': {'residues':   auth_asym_id auth_seq_id
    0            A        1354
    1            A        1397}},
  '5uak': {'pocket': {'residues':    auth_asym_id auth_seq_id
    0             A         347
    1             A         459
    2       

In [9]:
pd.to_pickle(results, "allofusion_results.pkl")